**Contexte**

Une entreprise possède plusieurs bâtiments équipés de capteurs IoT.
Chaque capteur collecte régulièrement des informations sur la température, l'humidité, la pression,
la consommation énergétique, le bâtiment, la date et l'heure de la mesure.
Chaque mesure possède également un état (OK, ALERTE et ERREUR).
L'objectif de l'atelier est de construire un modèle capable de prédire automatiquement l'état d'un
capteur à partir de ses mesures.
L'atelier suivra le workflow classique du Machine Learning :

`Dataset → Chargement → Exploration → Nettoyage → X / y → Train / Test → Prétraitement →
Modèle → fit()→ predict()→ Évaluation → Sauvegarde → Chargement → Réutilisation`

**Partie 0 – mise en place de l’environnement**

In [32]:
# importation des bibliothéques

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

Importation du dataset et vérification

In [26]:
df=pd.read_csv("../data/mesures_capteurs.csv")

df.head()

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 605 entries, 0 to 604
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_mesure     605 non-null    object 
 1   date_heure    605 non-null    object 
 2   id_capteur    605 non-null    object 
 3   batiment      605 non-null    object 
 4   temperature   599 non-null    float64
 5   humidite      600 non-null    float64
 6   pression      600 non-null    float64
 7   consommation  600 non-null    float64
 8   etat          601 non-null    object 
dtypes: float64(4), object(5)
memory usage: 42.7+ KB


In [28]:
df.describe(include="all")

,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
count,605,605,605,605,599.000000,600.00000,600.000000,600.000000,601
unique,600,600,12,4,NaN,NaN,NaN,NaN,3
top,M0302,2026-01-17 13:00:00,C002,B001,NaN,NaN,NaN,NaN,OK
freq,2,2,52,153,NaN,NaN,NaN,NaN,567
mean,NaN,NaN,NaN,NaN,24.878314,64.92620,1012.221900,208.675417,NaN
std,NaN,NaN,NaN,NaN,4.059576,10.76905,10.599042,72.243567,NaN
min,NaN,NaN,NaN,NaN,-18.500000,28.52000,850.000000,18.120000,NaN
25%,NaN,NaN,NaN,NaN,22.570000,58.17250,1006.790000,160.177500,NaN
50%,NaN,NaN,NaN,NaN,24.860000,65.37500,1012.855000,206.150000,NaN
75%,NaN,NaN,NaN,NaN,27.275000,71.61500,1017.827500,254.127500,NaN


**Partie 1 – Gestion des doublons**

1) vérifier l’existence de doublons dans df

In [33]:
# pour verifier l'existence de doublons on utilise la méthode duplicated la somme par sum 

nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons détectés : {nb_doublons}")

Nombre de doublons détectés : 5


2) supprimer les doublons puis vérifier la suppression

In [35]:
print("Ancienne dimension du dataframe", df.shape)

# la m"thode drop_duplicates permet de supprimer les doublons dans un dataset
df = df.drop_duplicates()


print(f"Nombre de doublons après nettoyage : {df.duplicated().sum()}")
print("Nouvelle dimension du dataframe :", df.shape)

Ancienne dimension du dataframe (605, 9)
Nombre de doublons après nettoyage : 0
Nouvelle dimension du dataframe : (600, 9)


Comme on fait des étude je vais en profiter également pour retirer les lignes dont la **cible `etat` est manquante** :
il est en effet impossible d'entraîner ou d'évaluer un modèle supervisé sur une observation dont on
ne connaît pas la vraie classe.

In [36]:
print("Valeurs manquantes dans 'etat' avant nettoyage :", df['etat'].isna().sum())
df = df.dropna(subset=['etat']).reset_index(drop=True)

print("Valeurs manquantes dans 'etat' après nettoyage :", df['etat'].isna().sum())
print("Dimension finale du dataframe :", df.shape)

Valeurs manquantes dans 'etat' avant nettoyage : 4
Valeurs manquantes dans 'etat' après nettoyage : 0
Dimension finale du dataframe : (596, 9)


**Partie 2 – Sélection de y (cible) et X (caractéristiques)**

1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite",
"pression" et "consommation" comme caractéristiques ou variables explicatives

In [ ]:
# Alors ici les variables explicatives sont "temperature", "humidite",
#"pression" et "consommation"qui correspond aus features 
# la variable cible est etat donc "etat" est le target_name

features = ["temperature", "humidite", "pression", "consommation"]
target = "etat"

X = df[features]
y = df[target]


2) Afficher les cinq premières lignes de X et de y

In [45]:
#pour afficher on vas utiliser head

print("Affichage de des cinq premiéres lignes de X")
print(X.head())
print("Affichage de des cinq premiéres lignes de y")

print(y.head())

Affichage de des cinq premiéres lignes de X
   temperature  humidite  pression  consommation
0        25.46     58.06   1008.95        287.28
1        24.00     79.73    993.39        116.20
2        25.82     54.47   1010.32        288.50
3        28.23     69.39   1019.62        136.65
4        20.58     53.80   1016.58        182.62
Affichage de des cinq premiéres lignes de y
0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: object


3) Quel est le type du problème de machine learning ?

Il s'agit d'un problème d'**apprentissage supervisé de classification**, et plus précisément de
**classification multi-classes** (3 classes possibles : `OK`, `ALERTE`, `ERREUR`) car :

- on dispose d'exemples étiquetés (la colonne `etat`) → **apprentissage supervisé** ;
- la variable à prédire est **catégorielle** (et non continue) → **classification** (et non régression) ;
- il y a plus de deux modalités possibles `OK`, `ALERTE`, `ERREUR` → **classification multi-classes** (et non binaire).

**Partie 3 – Découpage Train/Test**

Diviser X en deux ensembles distincts : un pour l'entraînement (train) et un pour le test (test).
Avec les conditions suivantes : 20% des données serviront au test ; garantir la reproductibilité du
découpage ; conserver les mêmes proportions de classes dans l'ensemble de train et de test que
dans les données d'origine.

In [47]:
# d'abord nous allons importer sickit-learn aprés importer le modele et enfin importer la fonction train_test_split 

from sklearn.model_selection import train_test_split


X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)
print("\nProportions dans y_train :")
print(y_train.value_counts(normalize=True))
print("\nProportions dans y_test :")
print(y_test.value_counts(normalize=True))

Taille X_train : (476, 4)
Taille X_test  : (120, 4)

Proportions dans y_train :
etat
OK        0.947479
ALERTE    0.042017
ERREUR    0.010504
Name: proportion, dtype: float64

Proportions dans y_test :
etat
OK        0.925
ALERTE    0.075
Name: proportion, dtype: float64


**Partie 4 – Gestion des valeurs manquantes**